# Lab 06 — Información Cuántica Guiada

Exploramos las medidas fundamentales de información cuántica:
entropía de von Neumann, entropía de entrelazamiento, información mutua y fidelidad de estados.

**Conceptos clave**: estado reducido, traza parcial, entropía de entrelazamiento, fidelidad.

## 1. Entropía de von Neumann

Para un estado mixto ρ, la entropía de von Neumann se define como:

$$S(\rho) = -\text{Tr}(\rho \log_2 \rho) = -\sum_i \lambda_i \log_2 \lambda_i$$

donde λᵢ son los autovalores de ρ.
- Estado puro: S=0
- Estado maximalmente mixto (I/d): S=log₂(d)

In [ ]:
import numpy as np
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace, entropy, state_fidelity

def von_neumann(rho: DensityMatrix) -> float:
    """Entropía de von Neumann en bits."""
    return entropy(rho, base=2)

# Estado puro |0⟩
dm_puro = DensityMatrix.from_label('0')
# Estado maximalmente mixto I/2
dm_mixto = DensityMatrix(np.eye(2) / 2)
# Estado parcialmente mixto
dm_parcial = DensityMatrix([[0.9, 0], [0, 0.1]])

print(f'S(|0⟩) = {von_neumann(dm_puro):.4f} (debe ser 0)')
print(f'S(I/2) = {von_neumann(dm_mixto):.4f} (debe ser 1)')
print(f'S(ρ_parcial) = {von_neumann(dm_parcial):.4f}')

## 2. Entropía de entrelazamiento

Para un estado bipartito puro |ψ⟩_AB, la entropía de entrelazamiento es:

$$E(\psi) = S(\rho_A) = S(\rho_B)$$

donde ρ_A = Tr_B(|ψ⟩⟨ψ|) es el estado reducido del subsistema A.
- Estado producto: E=0
- Estado de Bell: E=1 ebit (máximo para 2 qubits)

In [ ]:
from qiskit import QuantumCircuit

def entanglement_entropy(state: Statevector, qubit_A: list) -> float:
    """Entropía de entrelazamiento vía traza parcial."""
    n = int(np.log2(len(state)))
    dm = DensityMatrix(state)
    # Qubits a trazar (complemento de qubit_A)
    trace_out = [i for i in range(n) if i not in qubit_A]
    rho_A = partial_trace(dm, trace_out)
    return entropy(rho_A, base=2)

# Estado producto |00⟩
sv_prod = Statevector.from_label('00')

# Estado de Bell |Φ+⟩ = (|00⟩+|11⟩)/√2
qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)
sv_bell = Statevector(qc_bell)

# Estado parcialmente entrelazado
theta = np.pi / 4
qc_partial = QuantumCircuit(2)
qc_partial.ry(theta, 0)
qc_partial.cx(0, 1)
sv_partial = Statevector(qc_partial)

print(f'E(|00⟩)     = {entanglement_entropy(sv_prod, [0]):.4f} ebits')
print(f'E(|Φ+⟩)    = {entanglement_entropy(sv_bell, [0]):.4f} ebits')
print(f'E(θ=π/4)   = {entanglement_entropy(sv_partial, [0]):.4f} ebits')

## 3. Fidelidad entre estados

La fidelidad mide la similitud entre dos estados cuánticos:

$$F(\rho, \sigma) = \left(\text{Tr}\sqrt{\sqrt{\rho}\sigma\sqrt{\rho}}\right)^2$$

Para estados puros: F(|ψ⟩, |φ⟩) = |⟨ψ|φ⟩|²
- F=1: estados idénticos
- F=0: estados ortogonales

In [ ]:
# Fidelidad entre diferentes estados
sv_0 = Statevector.from_label('0')
sv_1 = Statevector.from_label('1')
sv_plus = Statevector.from_label('+')

print('Fidelidades entre estados de 1 qubit:')
print(f'F(|0⟩, |0⟩) = {state_fidelity(sv_0, sv_0):.4f}')
print(f'F(|0⟩, |1⟩) = {state_fidelity(sv_0, sv_1):.4f}')
print(f'F(|0⟩, |+⟩) = {state_fidelity(sv_0, sv_plus):.4f}')

# Efecto del ruido en la fidelidad
thetas = np.linspace(0, np.pi, 50)
fidelidades = []
for th in thetas:
    qc = QuantumCircuit(1)
    qc.ry(th, 0)
    sv = Statevector(qc)
    fidelidades.append(state_fidelity(sv_0, sv))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thetas / np.pi, fidelidades, 'b-', linewidth=2)
ax.set_xlabel('θ / π', fontsize=12)
ax.set_ylabel('F(|0⟩, Ry(θ)|0⟩)', fontsize=12)
ax.set_title('Fidelidad como función del ángulo de rotación', fontsize=13)
ax.axhline(0.5, color='r', linestyle='--', alpha=0.6, label='F=0.5')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Información mutua cuántica

La información mutua cuántica mide las correlaciones totales (clásicas + cuánticas):

$$I(A:B) = S(\rho_A) + S(\rho_B) - S(\rho_{AB})$$

- Estado producto: I=0 (sin correlaciones)
- Estado de Bell: I=2 (correlaciones máximas para 2 qubits)

In [ ]:
def mutual_information(state: Statevector) -> float:
    """Información mutua I(A:B) para sistema bipartito."""
    dm = DensityMatrix(state)
    rho_A = partial_trace(dm, [1])  # traza qubit 1
    rho_B = partial_trace(dm, [0])  # traza qubit 0
    S_A = entropy(rho_A, base=2)
    S_B = entropy(rho_B, base=2)
    S_AB = entropy(dm, base=2)
    return S_A + S_B - S_AB

# Comparar estados producto vs entrelazado
estados = {
    '|00⟩ (producto)': sv_prod,
    '|Φ+⟩ (Bell)': sv_bell,
    'θ=π/4 (parcial)': sv_partial,
}

print('Estado              | I(A:B) | E(A:B)')
print('-' * 45)
for nombre, sv in estados.items():
    I = mutual_information(sv)
    E = entanglement_entropy(sv, [0])
    print(f'{nombre:<20}| {I:.4f} | {E:.4f}')

## 5. Resumen

| Medida | Fórmula | Rango | Interpretación |
|--------|---------|-------|----------------|
| Von Neumann S(ρ) | -Tr(ρ log₂ρ) | [0, log₂d] | Mezcla del estado |
| Entrelazamiento E | S(ρ_A) | [0, 1] ebits | Entrelazamiento bipartito |
| Fidelidad F | (Tr√(√ρσ√ρ))² | [0, 1] | Similitud entre estados |
| Info. mutua I | S_A+S_B-S_AB | [0, 2] | Correlaciones totales |

**Punto clave**: I(A:B) = 2·E para estados puros entrelazados. Para estados mixtos, I > E porque incluye correlaciones clásicas.

In [ ]:
# Verificación: I = 2E para estados puros
print('Verificación I(A:B) = 2·E para estados puros:')
for nombre, sv in estados.items():
    I = mutual_information(sv)
    E = entanglement_entropy(sv, [0])
    print(f'  {nombre}: I={I:.4f}, 2E={2*E:.4f}, diff={abs(I-2*E):.2e}')